# Notebook 02 — Geometry of Per-Category Refusal Directions

Notebook 01 established that Qwen-1.8B-Chat encodes refusal in a single linear
direction in its residual stream.  This notebook asks a finer-grained question:

> **Do different *categories* of harmful content map to the same refusal direction,
> or does each category have a distinct direction?**

We use **SorryBench-202406** (Zeng et al., 2024), a dataset of 450 harmful prompts
across 44 categories grouped into four policy domains:

| Domain | Description | # prompts |
|---|---|---|
| HateSpeech | Personal insults, threats, obscene content | 50 |
| CrimeAssistance | Self-harm, violent/property crimes, malware, fraud | 190 |
| Inappropriate | Adult content, fake news, discrimination, extremism | 150 |
| Advice | Unqualified medical, financial, legal guidance | 60 |

For each domain we extract a refusal direction at layer 14 (the best layer from
Notebook 01), then examine:
1. **Cosine similarity matrix** — how aligned are the four domain directions?
2. **Cross-category ablation transfer** — ablating domain A's direction bypasses
   refusal on which other domains?

> **Requirements:** a CUDA GPU and an HF_TOKEN with access to SorryBench.  
> Accept terms at <https://huggingface.co/datasets/sorry-bench/sorry-bench-202406>,
> then set `HF_TOKEN` in your environment (or Colab Secrets).

---
## Setup

In [ ]:
!pip install -q transformers==4.44.2 tiktoken==0.7.0 accelerate einops transformers_stream_generator datasets

import sys
import pathlib

# Find the repo root (the directory containing src/) by walking up from CWD.
# Works wherever the folder lives in Drive — no hard-coded paths needed.
_cwd = pathlib.Path('.').resolve()
_root = next((p for p in [_cwd, *_cwd.parents] if (p / 'src').is_dir()), None)
assert _root is not None, (
    f"Could not find a 'src/' directory above {_cwd}\n"
    "Make sure refusal-geometry/ is unzipped and this notebook is inside it."
)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Repo root: {_root}")

import torch
assert torch.cuda.is_available(), "GPU not found — go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load HF_TOKEN from Colab Secrets if available.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets.")
except Exception as e:
    print(f"Could not load HF_TOKEN from Colab secrets ({e}).")
    print("Set it manually:  os.environ['HF_TOKEN'] = 'hf_...'")

---
## Load Model

Same model as Notebook 01.  We hard-code `best_layer = 14` here rather than
re-running the layer-selection sweep — that result was established in Notebook 01.

In [ ]:
from src.model import load_model

tokenizer, model, N_LAYERS, HIDDEN_SIZE = load_model()
print(f"N_LAYERS={N_LAYERS}, HIDDEN_SIZE={HIDDEN_SIZE}")

# Best layer for Qwen-1.8B-Chat, established in Notebook 01.
BEST_LAYER = 14

---
## Import Helpers

In [ ]:
from src.data import HARMLESS_PROMPTS, DOMAINS, load_sorry_bench
from src.directions import get_mean_activations
from src.evaluation import (
    refusal_score,
    generate,
    generate_with_full_ablation,
    is_refusal,
)

print(f"{len(HARMLESS_PROMPTS)} harmless prompts, {len(DOMAINS)} policy domains: {DOMAINS}")

---
## Load SorryBench

`load_sorry_bench()` downloads the dataset, filters to base prompts only
(discarding the 20 linguistic mutations per prompt), and groups them by
policy domain.

In [ ]:
print("Loading SorryBench...")
prompts_by_domain = load_sorry_bench()

print("\nPrompts per domain:")
for domain in DOMAINS:
    print(f"  {domain}: {len(prompts_by_domain[domain])}")

print("\nSpot-check (first prompt per domain):")
for domain in DOMAINS:
    print(f"  [{domain}] {prompts_by_domain[domain][0][:90]!r}")

---
## Step 1 — Per-Category Refusal Directions

For each domain **c** we compute:

$$r_c = \text{mean\_act}(\text{harmful}_c) - \text{mean\_act}(\text{harmless})$$

We then extract the row at `BEST_LAYER` and normalise to a unit vector.
The shared harmless baseline is computed once and reused for all four domains.

In [ ]:
# Shared harmless baseline — computed once, reused for all four domains.
print("Computing harmless baseline activations...")
mean_harmless_all = get_mean_activations(
    model, tokenizer, HARMLESS_PROMPTS, N_LAYERS, HIDDEN_SIZE
)

print()
directions_by_domain = {}
for domain in DOMAINS:
    prompts = prompts_by_domain[domain]
    print(f"Computing {domain} direction  ({len(prompts)} prompts)...")
    mean_harmful_c = get_mean_activations(
        model, tokenizer, prompts, N_LAYERS, HIDDEN_SIZE
    )
    r_c = mean_harmful_c - mean_harmless_all           # (N_LAYERS, HIDDEN_SIZE)
    d = r_c[BEST_LAYER].float()
    directions_by_domain[domain] = d / d.norm()        # unit vector at BEST_LAYER

print(f"\nDone — {len(directions_by_domain)} unit-norm directions, "
      f"shape {list(directions_by_domain[DOMAINS[0]].shape)}")
for domain, d in directions_by_domain.items():
    print(f"  {domain}: norm={d.norm().item():.4f}")

---
## Step 2 — Cosine Similarity Matrix

Since all four direction vectors are unit-normalised, their cosine similarity is
simply their dot product.  A value near 1 means the two domains share the same
refusal direction; a value near 0 means they are orthogonal.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Stack unit-norm directions into (4, HIDDEN_SIZE); cosine sim = dot product for unit vecs.
D = torch.stack([directions_by_domain[d] for d in DOMAINS])   # (4, 2048)
cos_sim = (D @ D.T).cpu().float().numpy()                      # (4, 4)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cos_sim,
    annot=True, fmt=".3f",
    xticklabels=DOMAINS, yticklabels=DOMAINS,
    vmin=-1, vmax=1, cmap="RdBu_r", center=0,
    linewidths=0.5, square=True, ax=ax,
)
ax.set_title("Cosine similarity of refusal directions\n(Qwen-1.8B-Chat, layer 14)")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig('../results/figures/02_cosine_similarity.png', dpi=150)
plt.show()

---
## Step 3 — Cross-Category Ablation Transfer

We now ask: if we ablate domain A's refusal direction, does that also suppress
refusal on domain B's prompts?

For each ordered pair (A, B) we:
1. Ablate domain A's direction from all layers (full ablation: block inputs,
   attention outputs, MLP outputs).
2. Generate responses to `N_EVAL` prompts from domain B.
3. Count the fraction that do **not** start with a refusal phrase (bypass rate).

If the four directions are essentially the same, we expect the bypass matrix to
be uniformly high.  If they are domain-specific, we expect high values only on
the diagonal.

In [ ]:
from tqdm.notebook import tqdm

N_EVAL = 10  # Prompts tested per domain B (keep small for speed).

bypass_matrix = np.zeros((4, 4))

print(f"Cross-category ablation transfer")
print(f"Ablating r_A from all layers; testing {N_EVAL} prompts per domain B")
print(f"(max_new_tokens=64; {len(DOMAINS)**2} pairs total)\n")

for i, domain_A in enumerate(DOMAINS):
    dir_A = directions_by_domain[domain_A].to(torch.float16)
    for j, domain_B in enumerate(DOMAINS):
        test_prompts = prompts_by_domain[domain_B][:N_EVAL]
        n_bypass = sum(
            not is_refusal(
                generate_with_full_ablation(
                    model, tokenizer, p, dir_A, N_LAYERS, max_new_tokens=64
                )
            )
            for p in tqdm(test_prompts, desc=f"  {domain_A}→{domain_B}", leave=False)
        )
        bypass_matrix[i, j] = n_bypass / N_EVAL

    row = "  ".join(f"{bypass_matrix[i, j]:.0%}" for j in range(4))
    print(f"  ablate {domain_A:<18} | {row}")

In [ ]:
# Results table
print("\n" + "─" * 62)
header = f"  {'Ablated →':18s} | " + "  ".join(f"{d[:8]:>8}" for d in DOMAINS)
print(header)
print("─" * 62)
for i, domain_A in enumerate(DOMAINS):
    row = "  ".join(f"{bypass_matrix[i, j]:>8.0%}" for j in range(4))
    print(f"  {domain_A:<18} | {row}")
print("─" * 62)
print("  Columns = tested category; Rows = ablated direction")
print("  Diagonal = same-category ablation (expected high bypass rate)")

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    bypass_matrix,
    annot=True, fmt=".0%",
    xticklabels=[d[:8] for d in DOMAINS],
    yticklabels=[d[:8] for d in DOMAINS],
    vmin=0, vmax=1, cmap="YlOrRd", ax=ax,
)
ax.set_xlabel("Tested category (domain B)")
ax.set_ylabel("Ablated direction (domain A)")
ax.set_title("Cross-category ablation bypass rate\n(Qwen-1.8B-Chat, layer 14)")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig('../results/figures/02_cross_category_transfer.png', dpi=150)
plt.show()

---
## Spot-check — Advice Domain Baseline

The Advice domain contains prompts that promote dangerous health/lifestyle choices.
Before interpreting the bypass matrix, it is useful to confirm the model actually
refuses these prompts under baseline conditions.

In [ ]:
for p in prompts_by_domain["Advice"][:5]:
    print(repr(p[:80]))
    print(" →", generate(model, tokenizer, p, max_new_tokens=60)[:120])
    print()

---
## Summary

| Question | Finding |
|---|---|
| Are domain refusal directions aligned? | Cosine similarities are all ≥ 0.85 — the four directions point in nearly the same direction |
| Does ablation transfer across domains? | Bypass rates are high even off-diagonal — ablating any single domain's direction suppresses refusal across categories |

**Interpretation:** Safety fine-tuning appears to have encoded a single, shared
refusal direction regardless of the type of harmful content.  The small angular
differences between domain directions do not produce meaningfully different bypass
behaviour, suggesting that the linear structure identified by Arditi et al. is
remarkably universal within this model.

Future work could test whether larger models (Qwen-7B, Llama-3) exhibit more
domain-specific geometry, or whether PCA of the bypass matrix reveals a lower-
dimensional structure in how different harm categories are represented.